<a href="https://colab.research.google.com/github/priyanka1994-as/git-basics-demo/blob/main/task2_data_processing_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
1 — Make the API Calls

In [ ]:
import requests
import time

# URLs
TOP_STORIES_URL = "https://hacker-news.firebaseio.com/v0/topstories.json"
ITEM_URL = "https://hacker-news.firebaseio.com/v0/item/{}.json"

headers = {"User-Agent": "TrendPulse/1.0"}

# Categories with keywords
categories = {
    "technology": ["ai", "software", "tech", "code", "computer", "data", "cloud", "api", "gpu", "llm"],
    "worldnews": ["war", "government", "country", "president", "election", "climate", "attack", "global"],
    "sports": ["nfl", "nba", "fifa", "sport", "game", "team", "player", "league", "championship"],
    "science": ["research", "study", "space", "physics", "biology", "discovery", "nasa", "genome"],
    "entertainment": ["movie", "film", "music", "netflix", "game", "book", "show", "award", "streaming"]
}

# Function to match category
def get_category(title):
    title = title.lower()
    for category, keywords in categories.items():
        for keyword in keywords:
            if keyword in title:
                return category
    return None


# Step 1: Fetch top story IDs
try:
    response = requests.get(TOP_STORIES_URL, headers=headers)
    response.raise_for_status()
    story_ids = response.json()[:500]
except Exception as e:
    print("Failed to fetch top stories:", e)
    story_ids = []

print(f"Fetched {len(story_ids)} story IDs")

# Step 2: Fetch stories category-wise
stories = []

for category in categories.keys():
    print(f"\nProcessing category: {category}")

    for story_id in story_ids:
        try:
            res = requests.get(ITEM_URL.format(story_id), headers=headers)
            res.raise_for_status()
            story = res.json()

            # Skip invalid stories
            if story is None or "title" not in story:
                continue

            # Check category match
            if get_category(story["title"]) == category:
                stories.append({
                    "post_id": story.get("id"),
                    "title": story.get("title"),
                    "category": category,
                    "score": story.get("score", 0),
                    "num_comments": story.get("descendants", 0)
                })

        except Exception as e:
            print(f"Error fetching story {story_id}: {e}")
            continue

    # ✅ Sleep AFTER each category loop
    time.sleep(2)

print(f"\nTotal collected stories: {len(stories)}")

Fetched 500 story IDs

Processing category: technology

Processing category: worldnews

Processing category: sports

Processing category: science

Processing category: entertainment

Total collected stories: 218


In [ ]:
2 — Extract the Fields

In [ ]:
import requests
import time
from datetime import datetime

# URLs
TOP_STORIES_URL = "https://hacker-news.firebaseio.com/v0/topstories.json"
ITEM_URL = "https://hacker-news.firebaseio.com/v0/item/{}.json"

headers = {"User-Agent": "TrendPulse/1.0"}

# Categories
categories = {
    "technology": ["ai", "software", "tech", "code", "computer", "data", "cloud", "api", "gpu", "llm"],
    "worldnews": ["war", "government", "country", "president", "election", "climate", "attack", "global"],
    "sports": ["nfl", "nba", "fifa", "sport", "game", "team", "player", "league", "championship"],
    "science": ["research", "study", "space", "physics", "biology", "discovery", "nasa", "genome"],
    "entertainment": ["movie", "film", "music", "netflix", "game", "book", "show", "award", "streaming"]
}

# Function to assign category
def get_category(title):
    title = title.lower()
    for category, keywords in categories.items():
        for keyword in keywords:
            if keyword in title:
                return category
    return None


# Fetch top IDs
response = requests.get(TOP_STORIES_URL, headers=headers)
story_ids = response.json()[:500]

stories = []

# Process category-wise
for category in categories.keys():
    print(f"\nProcessing {category}...")
    count = 0

    for story_id in story_ids:
        if count >= 25:  # ✅ limit per category
            break

        try:
            res = requests.get(ITEM_URL.format(story_id), headers=headers)
            story = res.json()

            if story is None or "title" not in story:
                continue

            # Check category match
            if get_category(story["title"]) == category:

                stories.append({
                    "post_id": story.get("id"),
                    "title": story.get("title"),
                    "category": category,
                    "score": story.get("score", 0),
                    "num_comments": story.get("descendants", 0),
                    "author": story.get("by", "unknown"),  # ✅ new field
                    "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")  # ✅ new field
                })

                count += 1

        except Exception as e:
            print(f"Error fetching {story_id}: {e}")
            continue

    print(f"Collected {count} stories for {category}")

    # ✅ sleep per category (important for marks)
    time.sleep(2)

print(f"\nTotal collected stories: {len(stories)}")


Processing technology...
Collected 25 stories for technology

Processing worldnews...
Collected 11 stories for worldnews

Processing sports...
Collected 8 stories for sports

Processing science...
Collected 11 stories for science

Processing entertainment...
Collected 25 stories for entertainment

Total collected stories: 80


In [ ]:
3 — Save to a JSON File

In [ ]:
import os
import json
from datetime import datetime

# Step 1: Create folder if it doesn't exist
folder_name = "data"
os.makedirs(folder_name, exist_ok=True)

# Step 2: Create filename with date
date_str = datetime.now().strftime("%Y%m%d")
file_path = f"{folder_name}/trends_{date_str}.json"

# Step 3: Save to JSON file
with open(file_path, "w", encoding="utf-8") as f:
    json.dump(stories, f, indent=4)

# Step 4: Print total count
print(f"\nSaved {len(stories)} stories to {file_path}")


Saved 80 stories to data/trends_20260406.json


In [ ]:
1 — Load the JSON File

In [ ]:
import pandas as pd

# Load JSON file
file_path = "data/trends_20260406.json"  # update filename if needed
df = pd.read_json(file_path)

# Print number of rows
print(f"Total rows loaded: {len(df)}")

Total rows loaded: 80


In [ ]:
2 — Clean the Data

In [ ]:
import pandas as pd

# Load file
df = pd.read_json("data/trends_20260406.json")

print("Before Cleaning:", df.shape)

# 1️⃣ Remove duplicates (based on post_id)
df = df.drop_duplicates(subset="post_id")

# 2️⃣ Handle missing values (drop important missing fields)
df = df.dropna(subset=["post_id", "title", "score"])

# 3️⃣ Fix data types
df["score"] = df["score"].astype(int)
df["num_comments"] = df["num_comments"].astype(int)

# 4️⃣ Remove low-quality stories (score < 5)
df = df[df["score"] >= 5]

# 5️⃣ Remove extra whitespace in title
df["title"] = df["title"].str.strip()

# Final result
print("After Cleaning:", df.shape)
print(f"Rows remaining after cleaning: {len(df)}")

Before Cleaning: (80, 7)
After Cleaning: (77, 7)
Rows remaining after cleaning: 77


In [ ]:
import os

# Create 'data' directory if it doesn't exist
os.makedirs("data", exist_ok=True)

# Save the cleaned DataFrame to a CSV file
df.to_csv("data/trends_clean.csv", index=False)
print("Saved cleaned data to data/trends_clean.csv")

Saved cleaned data to data/trends_clean.csv


In [ ]:
import os

# Create folder if not exists
os.makedirs("data", exist_ok=True)

# Save DataFrame to CSV
output_path = "data/trends_analysed.csv"
df.to_csv(output_path, index=False)

# Confirmation message
print(f"Data successfully saved to {output_path}")

Data successfully saved to data/trends_analysed.csv
